# Creating complex PDBe API queries

A practical, runnable tour of how to **combine several PDBe API v2 endpoints** into biological workflows.

We will work through three case studies, each chosen to illustrate a different *pattern* of API composition:

| # | Entry / accession | Pattern | Biological question |
|---|---|---|---|
| 1 | **1HVR** — HIV-1 protease | residue level joins (contacts ∩ interface ∩ conservation) | Which dimer interface residues contact the inhibitor, and how conserved are they? |
| 2 | **2DN2** — Hemoglobin (HbA) | entry → complex aggregation | Starting from one PDB entry, what does the whole PDBe-KB complex look like? |
| 3 | **Q96Y14** — *S. tokodaii* hexokinase | UniProt anchored aggregation (superposition clusters ∩ bound ligands) | Do PDB structures of one protein partition into conformational states that track ligand binding? |

**Base URL:** `https://www.ebi.ac.uk/pdbe/api/v2`

By the end of the tutorial you will have a small toolkit (one `fetch_json` helper plus a few
preview functions) and three reusable workflow templates expressed as pandas DataFrames.

## 2. Helper functions

We start with three small helpers that every workflow below reuses:

- `fetch_json(path)` — wraps `requests.get`, prints the URL and status, returns parsed JSON or `None`.
- `preview_keys(obj, n=8)` — prints the top level shape of a response without dumping the payload.
- `as_dataframe_or_none(records)` — builds a Dataframe from a list of dicts/tuples.

Keeping API access in a single helper lets the workflows below focus on the *biology* rather than on request handling.


In [1]:
import json
import requests
import pandas as pd
from textwrap import shorten

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 60)

BASE = "https://www.ebi.ac.uk/pdbe/api/v2"

def fetch_json(path, params=None, timeout=60, quiet=False, json_body=None):
    """GET (default) or POST (if json_body is given) {BASE}{path} and return parsed JSON, or None on failure."""
    url = BASE + path
    try:
        if json_body is not None:
            r = requests.post(url, json=json_body, params=params, timeout=timeout)
        else:
            r = requests.get(url, params=params, timeout=timeout)
    except requests.RequestException as exc:
        print(f"[NET ERR] {url} -> {exc}")
        return None
    if not quiet:
        print(f"GET {r.url}  ->  HTTP {r.status_code}  ({len(r.content):,} bytes)")
    if r.status_code != 200:
        print(f"  body: {r.text[:200]}")
        return None
    try:
        return r.json()
    except ValueError:
        print("  [WARN] response is not JSON")
        return None

def preview_keys(obj, label="response", n=8):
    """Print a compact summary of a JSON object without dumping the payload."""
    if obj is None:
        print(f"{label}: None"); return
    if isinstance(obj, dict):
        keys = list(obj.keys())
        print(f"{label}: dict with {len(keys)} key(s): {keys[:n]}{'...' if len(keys) > n else ''}")
    elif isinstance(obj, list):
        print(f"{label}: list of {len(obj)} item(s)")
        if obj and isinstance(obj[0], dict):
            print(f"  first item keys: {list(obj[0].keys())[:n]}")
    else:
        print(f"{label}: {type(obj).__name__}")

def as_dataframe_or_none(records, columns=None):
    """Build a DataFrame from a list of dicts (or list of tuples); returns empty DF if records is falsy."""
    if not records:
        return pd.DataFrame(columns=columns or [])
    return pd.DataFrame(records, columns=columns)

print("Helpers ready. BASE =", BASE)


Helpers ready. BASE = https://www.ebi.ac.uk/pdbe/api/v2


## 3. Example 1 — HIV-1 protease (1HVR)

**Story.** A cyclic urea inhibitor (XK-263, chem_comp_id `XK2`) sits on the dyad axis of the
HIV-1 protease homodimer. We want a table of residues that **both** contact the inhibitor
**and** lie on the dimer interface, annotated with **sequence conservation**.

**Endpoints used**

1. `/pdb/entry/summary/1hvr` — confirm the entry and assemblies.
2. `/pdb/entry/molecules/1hvr` — identify the protein entity and ligand entity.
3. `/pdb/bound_molecules/1hvr` — discover the bound molecule id (`bm_id`).
4. `/pdb/bound_molecule_interactions/1hvr/{bm_id}` — ligand contacts at residue level.
5. `/pisa/interfaces/1hvr/1` — all interfaces for assembly 1, with per residue data inline.
6. `/pdb/sequence_conservation/1hvr/{entity_id}` — conservation per position.

**Join key throughout this example:** `(chain_id, author_residue_number)`. For 1HVR the
author residue numbering of the protease is 1–99, which also matches the entity residue
index used by sequence conservation — so the joins are direct.


In [2]:
PDB_ID = "1hvr"
entry_summary = fetch_json(f"/pdb/entry/summary/{PDB_ID}")
entry_info = entry_summary[PDB_ID][0]
print("Title:", shorten(entry_info["title"], 100))
print("Method:", entry_info["experimental_method"])
print("Assemblies:")
for assembly in entry_info["assemblies"]:
    print(f"  assembly_id={assembly['assembly_id']:>2}  name={assembly['name']!r:>10}  preferred={assembly['preferred']}")


GET https://www.ebi.ac.uk/pdbe/api/v2/pdb/entry/summary/1hvr  ->  HTTP 200  (664 bytes)
Title: RATIONAL DESIGN OF POTENT, BIOAVAILABLE, NONPEPTIDE CYCLIC UREAS AS HIV PROTEASE INHIBITORS
Method: ['X-ray diffraction']
Assemblies:
  assembly_id= 1  name=   'dimer'  preferred=True


In [3]:
entry_molecules = fetch_json(f"/pdb/entry/molecules/{PDB_ID}")
molecule_rows = []
for molecule in entry_molecules[PDB_ID]:
    molecule_rows.append({
        "entity_id": molecule.get("entity_id"),
        "molecule_type": molecule.get("molecule_type"),
        "name": shorten("; ".join(molecule.get("molecule_name", []) or []), 60),
        "chains": ",".join(molecule.get("in_chains", []) or []),
        "length": molecule.get("length"),
        "n_copies": molecule.get("number_of_copies"),
    })
molecules_df = as_dataframe_or_none(molecule_rows)
molecules_df


GET https://www.ebi.ac.uk/pdbe/api/v2/pdb/entry/molecules/1hvr  ->  HTTP 200  (1,445 bytes)


,entity_id,molecule_type,name,chains,length,n_copies
0,1,polypeptide(L),Protease,"A,B",99.0,2
1,2,bound,[...],A,NaN,1


In [4]:
# entity 1 is the protein 
protein_entity_id = 1

bound_molecules = fetch_json(f"/pdb/bound_molecules/{PDB_ID}")
bound_molecule_rows = []
for bound_molecule in bound_molecules[PDB_ID]:
    for ligand in bound_molecule["composition"]["ligands"]:
        bound_molecule_rows.append({
            "bm_id": bound_molecule["bm_id"],
            "chem_comp_id": ligand["chem_comp_id"],
            "chain_id": ligand["chain_id"],
            "author_residue_number": ligand["author_residue_number"],
            "entity": ligand.get("entity"),
        })
bound_molecules_df = as_dataframe_or_none(bound_molecule_rows)
bound_molecules_df


GET https://www.ebi.ac.uk/pdbe/api/v2/pdb/bound_molecules/1hvr  ->  HTTP 200  (200 bytes)


,bm_id,chem_comp_id,chain_id,author_residue_number,entity
0,bm1,XK2,A,263,2


In [5]:
# Pick the first bound molecule (XK2 in 1HVR).
# target_bm_id = bm_df.loc[bm_df["chem_comp_id"] == "XK2", "bm_id"].iloc[0]
target_bm_id = 'bm1'
print("Target bm_id:", target_bm_id)

interactions_payload = fetch_json(f"/pdb/bound_molecule_interactions/{PDB_ID}/{target_bm_id}")
interaction_record = interactions_payload[PDB_ID][0]
print(f"Total interactions returned: {len(interaction_record['interactions'])}")

contact_rows = []
for interaction in interaction_record["interactions"]:
    end = interaction.get("end", {})
    if end.get("chem_comp_id") in (None, "HOH"):
        continue
    atom_subtypes = interaction.get("interactions", {})
    flat_subtypes = []
    for category, vals in atom_subtypes.items():
        for value in vals:
            flat_subtypes.append(f"{category}:{value}")
    contact_rows.append({
        "ligand_chem_comp_id": interaction["begin"]["chem_comp_id"],
        "chain_id": end["chain_id"],
        "author_residue_number": end["author_residue_number"],
        "residue_name": end["chem_comp_id"],
        "interaction_subtypes": ";".join(sorted(set(flat_subtypes))) or None,
    })
contacts_df = (as_dataframe_or_none(contact_rows)
               .drop_duplicates(subset=["chain_id", "author_residue_number"])
               .sort_values(["chain_id", "author_residue_number"])
               .reset_index(drop=True))
print(f"Unique residue contacts: {len(contacts_df)}")
contacts_df.head(10)


Target bm_id: bm1


GET https://www.ebi.ac.uk/pdbe/api/v2/pdb/bound_molecule_interactions/1hvr/bm1  ->  HTTP 200  (10,695 bytes)
Total interactions returned: 29
Unique residue contacts: 29


,ligand_chem_comp_id,chain_id,author_residue_number,residue_name,interaction_subtypes
0,XK2,A,23,LEU,atom_atom:hydrophobic
1,XK2,A,25,ASP,atom_atom:hbond;atom_atom:polar;atom_atom:vdw_clash;atom...
2,XK2,A,27,GLY,atom_atom:vdw_clash;atom_atom:weak_hbond;atom_atom:weak_...
3,XK2,A,28,ALA,atom_atom:hydrophobic;atom_atom:polar;atom_plane:CARBONPI
4,XK2,A,29,ASP,atom_atom:hydrophobic;atom_plane:DONORPI
5,XK2,A,30,ASP,atom_atom:hydrophobic;atom_atom:vdw;atom_atom:weak_polar
6,XK2,A,32,VAL,atom_atom:hydrophobic
7,XK2,A,47,ILE,atom_atom:hydrophobic;atom_plane:CARBONPI
8,XK2,A,48,GLY,atom_atom:weak_polar
9,XK2,A,49,GLY,atom_atom:vdw


In [6]:
# /pisa/interfaces/{pdb_id}/{assembly_id} returns ALL interfaces for the assembly.
# Iterating over them avoids hard-coding an interface_id (1HVR has one; other entries can have many).
interfaces_payload = fetch_json(f"/pisa/interfaces/{PDB_ID}/1")
assembly = interfaces_payload[PDB_ID]["assembly"]
print(f"assembly_id={interfaces_payload[PDB_ID]['assembly_id']}  "
      f"interface_count={assembly['interface_count']}")
for interface in assembly["interfaces"]:
    print(f"  interface_id={interface['interface_id']}  area={interface['interface_area']}  "
          f"n_int_residues={interface['number_interface_residues']}")

interface_rows = []
for interface in assembly["interfaces"]:
    interface_id = interface["interface_id"]
    for molecule in interface["molecules"]:
        chain = molecule["chain_id"]
        for residue_number, residue_name, bsa in zip(molecule["residue_seq_ids"],
                                   molecule["residue_label_comp_ids"],
                                   molecule["buried_surface_areas"]):
            bsa = float(bsa)
            if bsa <= 0:        # keep only residues that bury surface area at this interface
                continue
            interface_rows.append({
                "interface_id":           interface_id,
                "chain_id":               chain,
                "author_residue_number":  int(residue_number),
                "residue_name":           residue_name,
                "bsa":                    bsa,
            })
interface_df = (as_dataframe_or_none(interface_rows)
                .sort_values(["interface_id", "chain_id", "author_residue_number"])
                .reset_index(drop=True))
print(f"Interface residues collected (BSA > 0): {len(interface_df)}")
interface_df.head(10)

GET https://www.ebi.ac.uk/pdbe/api/v2/pisa/interfaces/1hvr/1  ->  HTTP 200  (55,393 bytes)
assembly_id=1  interface_count=1
  interface_id=1  area=1764.28  n_int_residues=99
Interface residues collected (BSA > 0): 85


,interface_id,chain_id,author_residue_number,residue_name,bsa
0,1,A,1,PRO,59.17
1,1,A,2,GLN,42.68
2,1,A,3,VAL,40.66
3,1,A,4,THR,16.23
4,1,A,5,LEU,112.49
5,1,A,6,TRP,52.14
6,1,A,7,GLN,5.43
7,1,A,8,ARG,55.19
8,1,A,9,PRO,23.77
9,1,A,11,VAL,4.69


### How the conservation score is computed

Before we read the numbers, one important point about what "conservation" means here.

**The alignment behind this score is not a single species alignment.**

1. `phmmer` searches the query against the **full UniProtKB database** — so the hits are homologues from *every species in UniProt*, not just (in this case) HIV-1 isolates.
2. An HMM is built from those hits with `hmmbuild`.
3. The query is aligned back to the HMM with `hmmalign`.
4. Per column probabilities and an information content score are computed.

So the signal is **family level, across species** — every homologue UniProtKB knows about — rather than within species variation. For interpretation we will use the **wild type amino acid probability** at each position: what fraction of the alignment column carries the residue actually present in 1HVR. A value of 1.0 means the residue is invariant across all homologues found; a low value means the position varies a lot across the family.

For HIV-1 protease the phmmer hits include HIV-1, HIV-2, SIV and many other retroviral and retroelement proteases — that breadth is why even the catalytic Asp25 ends up with a wild type probability of only ≈0.44 here.

In [7]:
conservation_response = fetch_json(f"/pdb/sequence_conservation/{PDB_ID}/{protein_entity_id}")
conservation_data = conservation_response["data"]

# Look up the wild-type AA probability at each position (i.e. the probability
# in the alignment column of the residue that is actually present in 1HVR).
protease_sequence = next(molecule["sequence"] for molecule in entry_molecules[PDB_ID]
                         if molecule["entity_id"] == protein_entity_id and molecule.get("sequence"))

conservation_rows = []
for position_index, residue_index in enumerate(conservation_data["index"]):
    wt_aa = protease_sequence[residue_index - 1]
    wt_prob_key = f"probability_{wt_aa}"
    wt_prob = conservation_data.get(wt_prob_key, [None] * len(conservation_data["index"]))[position_index]
    conservation_rows.append({
        "entity_residue_index": residue_index,
        "wt_aa": wt_aa,
        "wt_probability": round(wt_prob, 3) if wt_prob is not None else None,
    })
conservation_df = as_dataframe_or_none(conservation_rows)
print(f"Conservation positions returned: {len(conservation_df)}  (sequence length: {len(protease_sequence)})")
conservation_df.head(10)

GET https://www.ebi.ac.uk/pdbe/api/v2/pdb/sequence_conservation/1hvr/1  ->  HTTP 200  (12,915 bytes)
Conservation positions returned: 99  (sequence length: 99)


,entity_residue_index,wt_aa,wt_probability
0,1,P,0.711
1,2,Q,0.257
2,3,V,0.135
3,4,T,0.260
4,5,L,0.333
5,6,W,0.230
6,7,Q,0.227
7,8,R,0.327
8,9,P,0.460
9,10,L,0.232


In [8]:
# Compose the final table: contacts ∩ interface residues, annotated with conservation.
annotated_contacts = (contacts_df
           .merge(interface_df[["interface_id", "chain_id", "author_residue_number", "bsa"]],
                  on=["chain_id", "author_residue_number"], how="inner")
           .merge(conservation_df,
                  left_on="author_residue_number",
                  right_on="entity_residue_index", how="left"))

ligand_label = molecules_df.loc[
    molecules_df["entity_id"] == bound_molecules_df.loc[bound_molecules_df["bm_id"] == target_bm_id, "entity"].iloc[0],
    "name",
].iloc[0]

interface_contact_table = pd.DataFrame({
    "pdb_id":                  PDB_ID,
    "assembly_id":             1,
    "interface_id":            annotated_contacts["interface_id"],
    "chain_id":                annotated_contacts["chain_id"],
    "residue_number":          annotated_contacts["author_residue_number"],
    "residue_name":            annotated_contacts["residue_name"],
    "ligand_id":               annotated_contacts["ligand_chem_comp_id"],
    "ligand_name":             ligand_label,
    "ligand_interaction_type": annotated_contacts["interaction_subtypes"],
    "bsa":                     annotated_contacts["bsa"].round(1),
    "wt_probability":          annotated_contacts["wt_probability"],
}).sort_values(["interface_id", "chain_id", "residue_number"]).reset_index(drop=True)

print(f"Ligand-contacting interface residues: {len(interface_contact_table)}")
interface_contact_table

Ligand-contacting interface residues: 21


,pdb_id,assembly_id,interface_id,chain_id,residue_number,residue_name,ligand_id,ligand_name,ligand_interaction_type,bsa,wt_probability
0,1hvr,1,1,A,23,LEU,XK2,[...],atom_atom:hydrophobic,16.2,0.446
1,1hvr,1,1,A,25,ASP,XK2,[...],atom_atom:hbond;atom_atom:polar;atom_atom:vdw_clash;atom...,14.4,0.443
2,1hvr,1,1,A,27,GLY,XK2,[...],atom_atom:vdw_clash;atom_atom:weak_hbond;atom_atom:weak_...,45.6,0.518
3,1hvr,1,1,A,29,ASP,XK2,[...],atom_atom:hydrophobic;atom_plane:DONORPI,35.4,0.448
4,1hvr,1,1,A,32,VAL,XK2,[...],atom_atom:hydrophobic,3.4,0.388
5,1hvr,1,1,A,47,ILE,XK2,[...],atom_atom:hydrophobic;atom_plane:CARBONPI,11.8,0.319
6,1hvr,1,1,A,48,GLY,XK2,[...],atom_atom:weak_polar,1.1,0.235
7,1hvr,1,1,A,49,GLY,XK2,[...],atom_atom:vdw,28.9,0.464
8,1hvr,1,1,A,50,ILE,XK2,[...],atom_atom:hbond;atom_atom:hydrophobic;atom_atom:polar;at...,135.2,0.439
9,1hvr,1,1,A,81,PRO,XK2,[...],atom_atom:hydrophobic,35.0,0.445


**Reading the table.** We joined three independent endpoints — ligand contacts, PISA interface residues, and per position sequence conservation — on `(chain_id, author_residue_number)` to produce a single residue level table for the XK-263 / HIV-1 protease complex. The catalytic Asp25 appears in the contact list, which is the expected sanity check for a protease inhibitor binding the active site.

The point of this example is the *pattern* rather than any specific biological claim: a residue level join across three PDBe endpoints, expressed as one pandas DataFrame.

**Takeaway.** Every residue contacting the inhibitor has `wt_probability` in the range 0.25–0.55, including the catalytic Asp25 at ≈0.44. Because the alignment behind this number is across **UniProtKB homologues from many species** (not HIV-1 isolates only), these positions sit in columns where the broader retroviral protease family has already tolerated substitution — i.e. the inhibitor contacts residues that vary across the family.

## 4. Example 2 — Hemoglobin HbA (2DN2 → PDB-CPX-154652)

**Story.** From a single PDB entry of hemoglobin, we want to:

1. discover its **PDBe Complex ID**,
2. list **participants** at the complex level and **assemblies**,
3. summarise the **ligand landscape** across all PDB entries of this complex,
4. enumerate **subcomplexes and supercomplexes**.

**Endpoints used**

1. `/complex/details/2dn2?id_type=pdb_id` — find the PDB Complex id.
2. `/complex/details/{pdb_complex_id}?id_type=pdb_complex_id` — full complex record.
3. `/complex/bound_molecules_summary/{pdb_complex_id}` — ligand landscape.
4. `/complex/interactions/{pdb_complex_id}` — sub/supercomplexes.

**Pattern.** "Lift" from one PDB entry to the family of entries that share the same biological
complex, then aggregate.

In [9]:
SOURCE_PDB = "2dn2"
complex_details_by_pdb = fetch_json(f"/complex/details/{SOURCE_PDB}", params={"id_type": "pdb_id"})
complex_match = complex_details_by_pdb[SOURCE_PDB][0]
pdb_complex_id = complex_match["pdb_complex_id"]
print(f"PDB Complex id for {SOURCE_PDB}: {pdb_complex_id}")
print(f"Complex name: {complex_match['name']}")
print(f"Complex Portal id: {complex_match.get('complex_portal_id')}")
print(f"Sub-complexes: {len(complex_match['subcomplexes'])}, super-complexes: {len(complex_match['supercomplexes'])}")


GET https://www.ebi.ac.uk/pdbe/api/v2/complex/details/2dn2?id_type=pdb_id  ->  HTTP 200  (491 bytes)
PDB Complex id for 2dn2: PDB-CPX-154652
Complex name: Hemoglobin HbA complex
Complex Portal id: CPX-2158
Sub-complexes: 0, super-complexes: 11


In [10]:
complex_details_response = fetch_json(f"/complex/details/{pdb_complex_id}", params={"id_type": "pdb_complex_id"})
complex_record = complex_details_response[pdb_complex_id][0]

complex_details = pd.DataFrame([{
    "pdb_complex_id":       pdb_complex_id,
    "complex_name":         complex_record["name"],
    "complex_portal_id":    complex_record.get("complex_portal_id"),
    "source_organism":      complex_record.get("source_organism"),
    "oligomeric_state":     complex_record.get("oligomeric_state"),
    "total_chains":         complex_record.get("total_chains"),
    "polymer_composition":  complex_record.get("polymer_composition"),
    "representative_pdb_id": complex_record.get("representative_structure", {}).get("pdb_id"),
    "representative_assembly_id": complex_record.get("representative_structure", {}).get("assembly_id"),
    "number_of_assemblies": len(complex_record.get("assemblies", [])),
}])
complex_details.T


GET https://www.ebi.ac.uk/pdbe/api/v2/complex/details/PDB-CPX-154652?id_type=pdb_complex_id  ->  HTTP 200  (99,541 bytes)


,0
pdb_complex_id,PDB-CPX-154652
complex_name,Hemoglobin HbA complex
complex_portal_id,CPX-2158
source_organism,Homo sapiens
oligomeric_state,Heterotetramer
total_chains,4
polymer_composition,Protein(4) complex
representative_pdb_id,2d5z
representative_assembly_id,1
number_of_assemblies,342


In [11]:
participants_df = as_dataframe_or_none(complex_record.get("participants", []))
participants_df


,accession,stoichiometry,accession_type,name,gene_name
0,P69905,2,UniProt,Hemoglobin subunit alpha,HBA1
1,P68871,2,UniProt,Hemoglobin subunit beta,HBB


In [12]:
bound_molecules_summary = fetch_json(f"/complex/bound_molecules_summary/{pdb_complex_id}")
ligand_rows = []
for chem_comp_id, ligand_info in bound_molecules_summary[pdb_complex_id].items():
    ligand_rows.append({
        "ligand_id":            chem_comp_id,
        "ligand_name":          shorten(ligand_info.get("name", ""), 50),
        "ligand_annotations":   ", ".join(ligand_info.get("annotations") or []) or None,
        "num_pdb_entries":      ligand_info.get("num_pdb_entries"),
        "num_chains":           ligand_info.get("num_chains"),
        "num_ligand_instances": ligand_info.get("num_ligand_instances"),
    })
ligand_summary = (as_dataframe_or_none(ligand_rows)
                  .sort_values("num_pdb_entries", ascending=False)
                  .reset_index(drop=True))
print(f"Distinct ligand types observed across the HbA complex: {len(ligand_summary)}")
ligand_summary.head(15)


GET https://www.ebi.ac.uk/pdbe/api/v2/complex/bound_molecules_summary/PDB-CPX-154652  ->  HTTP 200  (12,350 bytes)
Distinct ligand types observed across the HbA complex: 90


,ligand_id,ligand_name,ligand_annotations,num_pdb_entries,num_chains,num_ligand_instances
0,HEM,PROTOPORPHYRIN IX CONTAINING FE,Cofactor-like,294,1027,1148
1,CMO,CARBON MONOXIDE,None,85,254,326
2,OXY,OXYGEN MOLECULE,None,41,146,156
3,SO4,SULFATE ION,None,28,66,115
4,PO4,PHOSPHATE ION,None,19,28,41
5,HNI,PROTOPORPHYRIN IX CONTAINING NI(II),None,18,42,42
6,GOL,GLYCEROL,None,14,28,47
7,MBN,TOLUENE,None,12,14,53
8,2FU,BUT-2-ENEDIAL,None,12,12,12
9,NO,NITRIC OXIDE,None,6,20,22


In [13]:
complex_interactions = fetch_json(f"/complex/interactions/{pdb_complex_id}")
relation_rows = []
for relation in complex_interactions[pdb_complex_id]:
    relation_rows.append({
        "relation_type":            relation.get("relationship_type"),
        "related_complex_id":       relation.get("pdb_complex_id"),
        "related_complex_name":     shorten(relation.get("name", ""), 60),
        "additional_components":    ", ".join(f"{participant['name']} ({participant['stoichiometry']})"
                                            for participant in relation.get("additional_participants", [])),
        "representative_pdb_id":    relation.get("representative_structure", {}).get("pdb_id"),
    })
related_complexes = as_dataframe_or_none(relation_rows)
print(f"Related complexes returned: {len(related_complexes)}")
related_complexes.head(10)


GET https://www.ebi.ac.uk/pdbe/api/v2/complex/interactions/PDB-CPX-154652  ->  HTTP 200  (7,241 bytes)
Related complexes returned: 11


,relation_type,related_complex_id,related_complex_name,additional_components,representative_pdb_id
0,super-complex,PDB-CPX-121255,Hemoglobin HbA complex,"Haptoglobin (2), Haptoglobin-hemoglobin receptor (2), Ir...",4wjg
1,super-complex,PDB-CPX-154653,Hemoglobin HbA complex and Iron-regulated surface [...],Iron-regulated surface determinant protein H (4),4ij2
2,super-complex,PDB-CPX-154654,Hemoglobin HbA complex and Membrane protein,Membrane protein (2),9bcj
3,super-complex,PDB-CPX-154656,Hemoglobin HbA complex and Iron-regulated surface [...],Iron-regulated surface determinant protein H (2),4fc3
4,super-complex,PDB-CPX-154657,Hemoglobin HbA complex and Iron-regulated surface [...],Iron-regulated surface determinant protein B (1),7pcq
5,super-complex,PDB-CPX-154658,Hemoglobin HbA complex and Iron-regulated surface [...],Iron-regulated surface determinant protein B (2),7pch
6,super-complex,PDB-CPX-154659,Hemoglobin HbA complex and Streptococcal hemoprotein [...],Streptococcal hemoprotein receptor (1),8dov
7,super-complex,PDB-CPX-154660,Hemoglobin HbA complex and Streptococcal hemoprotein [...],Streptococcal hemoprotein receptor (2),8dov
8,super-complex,PDB-CPX-154661,Hemoglobin HbA complex and Streptococcal hemoprotein [...],Streptococcal hemoprotein receptor (3),7cue
9,super-complex,PDB-CPX-203292,Hemoglobin HbA complex and Iron-regulated surface [...],Iron-regulated surface determinant protein H (3),9s3p


**What we just did.** With four API calls we lifted a single PDB id into a complex level summary spanning hundreds of structures, distinguished ligands annotated as `Cofactor-like` versus `Drug-like`, and enumerated the haptoglobin / IsdH supercomplexes — all keyed off the discovered `pdb_complex_id`.

## 5. Example 3 — Conformational states via UniProt superposition (Q96Y14)

**Story.** A single UniProt accession often maps to many PDB structures. We want to ask whether those structures occupy distinct **conformational states**, and — if they do — whether the state membership tracks the **bound ligand state** of each chain.

The protein here is *Sulfolobus tokodaii* hexokinase (UniProt **Q96Y14**, 299 aa). PDBe-KB's **superposition** endpoint precomputes a structural clustering of all related PDB chains, so we can read the conformational groups directly off the API rather than running our own structure alignment.

**Endpoints used**

1. `/uniprot/superposition/{accession}` — discover the conformational clusters.
2. `/pdb/entry/summary/{pdb_id}` — entry title and experimental method per cluster member.
3. `/pdb/entry/molecules/{pdb_id}` — `mutation_flag` and modified residue sanity checks.
4. `/pdb/bound_molecules/{pdb_id}` — ligand inventory per chain.
5. `/pdb/compound/summary/{chem_comp_id}` — chemistry metadata (name, formula, synonyms) for each ligand encountered, so the audience does not need to recognise PDB three letter codes.

**Pattern.** From one UniProt accession, fan out to every superimposed PDB chain, group by structural cluster, and annotate each chain with its bound ligand state.

In [14]:
uniprot_accession = "Q96Y14"
superposition = fetch_json(f"/uniprot/superposition/{uniprot_accession}")[uniprot_accession]
print(f"Segments returned: {len(superposition)}")
for segment in superposition:
    print(f"  segment_start={segment['segment_start']}  segment_end={segment['segment_end']}  "
          f"clusters={len(segment['clusters'])}")

cluster_rows = []
for segment in superposition:
    for cluster_id, members in enumerate(segment["clusters"]):
        for member in members:
            cluster_rows.append({
                "cluster_id":        cluster_id,
                "pdb_id":            member["pdb_id"],
                "auth_asym_id":      member["auth_asym_id"],
                "entity_id":         member["entity_id"],
                "is_representative": member["is_representative"],
            })
clusters_df = (as_dataframe_or_none(cluster_rows)
               .sort_values(["cluster_id", "pdb_id", "auth_asym_id"])
               .reset_index(drop=True))
print(f"Cluster-chain rows: {len(clusters_df)}  "
      f"(across {clusters_df['pdb_id'].nunique()} distinct PDB entries)")
clusters_df

GET https://www.ebi.ac.uk/pdbe/api/v2/uniprot/superposition/Q96Y14  ->  HTTP 200  (751 bytes)
Segments returned: 1
  segment_start=1  segment_end=299  clusters=2
Cluster-chain rows: 7  (across 4 distinct PDB entries)


,cluster_id,pdb_id,auth_asym_id,entity_id,is_representative
0,0,2e2o,A,1,True
1,0,2e2q,A,1,False
2,1,2e2n,A,1,False
3,1,2e2n,B,1,False
4,1,2e2p,A,1,False
5,1,2e2p,B,1,False
6,1,2e2q,B,1,True


In [15]:
# One summary + one molecules call per unique PDB id (here N=4).
pdb_ids = sorted(clusters_df["pdb_id"].unique())
entry_metadata_rows = []
for pdb_id in pdb_ids:
    summary_response = fetch_json(f"/pdb/entry/summary/{pdb_id}", quiet=True)
    title  = summary_response[pdb_id][0]["title"] if summary_response and pdb_id in summary_response else None
    method = ",".join(summary_response[pdb_id][0].get("experimental_method") or []) if summary_response else None
    molecules_response = fetch_json(f"/pdb/entry/molecules/{pdb_id}", quiet=True)
    protein = next((x for x in (molecules_response or {}).get(pdb_id, [])
                    if x.get("molecule_type") == "polypeptide(L)"), {})
    modified_positions = list((protein.get("pdb_sequence_indices_with_multiple_residues") or {}).keys())
    entry_metadata_rows.append({
        "pdb_id":         pdb_id,
        "title":          shorten(title or "", 80),
        "method":         method,
        "mutation_flag":  protein.get("mutation_flag"),
        "modified_residue_positions": ",".join(modified_positions) if modified_positions else None,
    })
entry_metadata_df = as_dataframe_or_none(entry_metadata_rows)
entry_metadata_df

,pdb_id,title,method,mutation_flag,modified_residue_positions
0,2e2n,Crystal structure of Sulfolobus tokodaii hexokinase in t...,X-ray diffraction,None,None
1,2e2o,Crystal structure of Sulfolobus tokodaii hexokinase in c...,X-ray diffraction,None,None
2,2e2p,Crystal structure of Sulfolobus tokodaii hexokinase in c...,X-ray diffraction,None,None
3,2e2q,Crystal structure of Sulfolobus tokodaii hexokinase in c...,X-ray diffraction,None,None


In [16]:
# Bound molecules per chain across the cluster members.
ligand_rows = []
for pdb_id in pdb_ids:
    bound_molecules = fetch_json(f"/pdb/bound_molecules/{pdb_id}", quiet=True)
    for bound_molecule in (bound_molecules or {}).get(pdb_id, []):
        for ligand in bound_molecule["composition"]["ligands"]:
            ligand_rows.append({
                "pdb_id":       pdb_id,
                "auth_asym_id": ligand["chain_id"],
                "chem_comp_id": ligand["chem_comp_id"],
            })
ligand_per_chain = (as_dataframe_or_none(ligand_rows)
                    .groupby(["pdb_id", "auth_asym_id"])["chem_comp_id"]
                    .agg(lambda s: ",".join(sorted(set(s))))
                    .reset_index()
                    .rename(columns={"chem_comp_id": "bound_chem_comp_ids"}))
ligand_per_chain

,pdb_id,auth_asym_id,bound_chem_comp_ids
0,2e2n,A,SO4
1,2e2n,B,"EPE,SO4"
2,2e2o,A,BGC
3,2e2o,A_2,BGC
4,2e2p,A,"ADP,SO4"
5,2e2p,B,"ADP,EPE,SO4"
6,2e2q,A,"ADP,MG,XYP"
7,2e2q,B,ADP


In [17]:
# Compose the cluster table without imposing any chemistry labels:
# show which chain belongs to which cluster + which ligands are modelled on each chain.
cluster_ligand_table = (clusters_df
                .merge(entry_metadata_df, on="pdb_id", how="left")
                .merge(ligand_per_chain, on=["pdb_id", "auth_asym_id"], how="left"))
cluster_ligand_table = cluster_ligand_table[[
    "cluster_id", "pdb_id", "auth_asym_id", "is_representative",
    "title", "mutation_flag", "modified_residue_positions",
    "bound_chem_comp_ids",
]].sort_values(["cluster_id", "pdb_id", "auth_asym_id"]).reset_index(drop=True)

# Let the API surface the pattern: for each cluster, which ligand chem_comp_ids are
# unique to that cluster and which are shared with the other?  No prior knowledge of
# the protein's chemistry is required.
ligands_long = (cluster_ligand_table
             .assign(ligand=cluster_ligand_table["bound_chem_comp_ids"].fillna("").str.split(","))
             .explode("ligand"))
ligands_long = ligands_long[ligands_long["ligand"] != ""]
ligands_by_cluster = ligands_long.groupby("cluster_id")["ligand"].agg(set).to_dict()

print(f"Chains in clustering: {len(cluster_ligand_table)}\n")
print("Distinguishing ligands per cluster (set differences across clusters):")
for cluster_id, ligands in ligands_by_cluster.items():
    others = set().union(*(v for k, v in ligands_by_cluster.items() if k != cluster_id))
    unique = sorted(ligands - others)
    shared = sorted(ligands & others)
    print(f"  cluster {cluster_id}:  unique = {unique}    shared = {shared}")

cluster_ligand_table

Chains in clustering: 7

Distinguishing ligands per cluster (set differences across clusters):
  cluster 0:  unique = ['BGC', 'MG', 'XYP']    shared = ['ADP']
  cluster 1:  unique = ['EPE', 'SO4']    shared = ['ADP']


,cluster_id,pdb_id,auth_asym_id,is_representative,title,mutation_flag,modified_residue_positions,bound_chem_comp_ids
0,0,2e2o,A,True,Crystal structure of Sulfolobus tokodaii hexokinase in c...,None,None,BGC
1,0,2e2q,A,False,Crystal structure of Sulfolobus tokodaii hexokinase in c...,None,None,"ADP,MG,XYP"
2,1,2e2n,A,False,Crystal structure of Sulfolobus tokodaii hexokinase in t...,None,None,SO4
3,1,2e2n,B,False,Crystal structure of Sulfolobus tokodaii hexokinase in t...,None,None,"EPE,SO4"
4,1,2e2p,A,False,Crystal structure of Sulfolobus tokodaii hexokinase in c...,None,None,"ADP,SO4"
5,1,2e2p,B,False,Crystal structure of Sulfolobus tokodaii hexokinase in c...,None,None,"ADP,EPE,SO4"
6,1,2e2q,B,True,Crystal structure of Sulfolobus tokodaii hexokinase in c...,None,None,ADP


In [18]:
# Annotate every distinguishing chem_comp_id with metadata from the API itself.
# One batched POST instead of N GETs — body is a JSON-encoded comma-separated hetcode list.
ligand_ids = sorted(set(ligands_long["ligand"]))
print(f"Looking up {len(ligand_ids)} unique ligand(s) in one POST: {ligand_ids}")

compound_summary_batch = fetch_json("/pdb/compound/summary",
                       json_body=",".join(ligand_ids))

legend_rows = []
for chem_comp_id in ligand_ids:
    compound_record = (compound_summary_batch or {}).get(chem_comp_id) or {}
    if isinstance(compound_record, list):
        compound_record = compound_record[0] if compound_record else {}
    synonyms = compound_record.get("synonyms") or []
    short_syn = next((synonym.get("value") for synonym in synonyms
                      if synonym.get("value") and len(synonym["value"]) <= 12), None)
    legend_rows.append({
        "chem_comp_id":   chem_comp_id,
        "name":           shorten(compound_record.get("name") or "", 50),
        "formula":        compound_record.get("formula"),
        "weight":         compound_record.get("weight"),
        "common_synonym": short_syn,
    })
ligand_legend = as_dataframe_or_none(legend_rows)
ligand_legend

Looking up 6 unique ligand(s) in one POST: ['ADP', 'BGC', 'EPE', 'MG', 'SO4', 'XYP']


GET https://www.ebi.ac.uk/pdbe/api/v2/pdb/compound/summary  ->  HTTP 200  (20,517 bytes)


,chem_comp_id,name,formula,weight,common_synonym
0,ADP,ADENOSINE-5'-DIPHOSPHATE,C10 H15 N5 O10 P2,427.201,ADP
1,BGC,beta-D-glucopyranose,C6 H12 O6,180.156,β-D-glucose
2,EPE,4-(2-HYDROXYETHYL)-1-PIPERAZINE [...],C8 H18 N2 O4 S,238.305,HEPES
3,MG,MAGNESIUM ION,Mg,24.305,None
4,SO4,SULFATE ION,O4 S,96.063,Sulfate
5,XYP,beta-D-xylopyranose,C5 H10 O5,150.130,xylose


**Reading the table.** The chains split into two structural clusters, and the per cluster set difference of bound `chem_comp_id`s — computed directly from the API output, with no prior knowledge of this protein's chemistry — surfaces the distinguishing ligands:

- Cluster 0 unique ligands: `BGC`, `XYP`, `MG`.
- Cluster 1 unique ligands: `SO4`, `EPE`.
- Shared across clusters: `ADP`.

The ligand legend table above (built from `/pdb/compound/summary/{chem_comp_id}`) decodes those identifiers without us having to look anything up by hand: cluster 0 is defined by two sugars (β-D-glucopyranose, β-D-xylopyranose) plus a divalent cation (Mg²⁺); cluster 1 is defined by crystallisation additives (sulfate, HEPES). ADP appears in both clusters and therefore doesn't drive the split.

The most informative case is **2e2q chain B**: it shares a crystal with 2e2q chain A, but no sugar is modelled in chain B, and the two chains fall into different clusters — two protomers in one asymmetric unit captured in two different conformational states.

Two negative controls reinforce the conclusion:

- `mutation_flag` is unset for every entry → not a sequence variant artefact.
- `modified_residue_positions` is empty → no chemical modification artefacts.

Thus one UniProt accession yields four PDB entries, seven rows (one per (cluster, chain) pair), and a conformational pattern that is both surfaced *and labelled* by the API itself, without encoding the protein's chemistry a priori.

## 6. Summary — three patterns for complex PDBe API workflows

| Pattern | Where to use it | Endpoints chained | Join key |
|---|---|---|---|
| **Residue level intersection** (Example 1) | "Which residues do X *and* Y?" — contacts + interface + conservation | `entry/molecules` → `bound_molecules` → `bound_molecule_interactions` → `pisa/interfaces` → `sequence_conservation` | `(chain_id, author_residue_number)` and the entity residue index |
| **Entry to complex aggregation** (Example 2) | Lift one PDB entry into all entries sharing the same biological complex | `complex/details (by pdb_id)` → `complex/details (by pdb_complex_id)` → `complex/bound_molecules_summary` → `complex/interactions` | `pdb_complex_id` |
| **UniProt anchored aggregation** (Example 3) | From one UniProt accession, group every related PDB chain by structural state | `uniprot/superposition` → `pdb/entry/summary` → `pdb/entry/molecules` → `pdb/bound_molecules` → `pdb/compound/summary` | `(pdb_id, auth_asym_id)` |

**Practical reminders from the live API**

- `/pdb/sequence_conservation/{pdb_id}/{entity_id}` takes **`entity_id`**, not `assembly_id`.
- `/pdb/bound_molecule_interactions/{pdb_id}/{bm_id}` — the path parameter is `bm_id`.
- `/pisa/interfaces/{pdb_id}/{assembly_id}` returns *all* interfaces for an assembly; each interface carries per residue data in `molecules[].residue_*` parallel arrays — so you can iterate rather than hardcoding an `interface_id`.
- `/uniprot/superposition/{accession}` returns a list of segments; each segment has a `clusters` field that is a list of lists — the outer index is the cluster id, inner items are the chain memberships.

These three composition patterns cover most of what you will need to build interactive analyses on top of the PDBe API.